<a href="https://colab.research.google.com/github/pnngocdoan/agent-history-effects/blob/main/notebooks/run_main_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rule erosion: main study

To run your model:
1. Put your name as DRI on a row of the model table in the research doc.
2. Runtime → Change runtime type → a GPU (L4 or A100 for anything that is not Qwen3).
3. In **cell 4**, keep only your model in `MODELS` (or add it), then Runtime → Run all.

For each model the notebook starts it, runs a 3-conversation pilot and skips the model if
anything looks broken, then runs k = 0 and k = 10. Everything is resumable: if the runtime dies,
run the cells again and only missing conversations are run. Results are pushed to GitHub every
10 minutes and after every model, and the model shows up in `reports/viewer.html` under its
family on its own.

You need, once: push access to this repo and a GitHub token; for gated models (e.g. Gemma), a
Hugging Face token after accepting the model's license on its page.

In [ ]:
# 1. setup (once per runtime)
import os, sys
REPO = '/content/agent-history-effects'
if not os.path.exists(REPO):
    !git clone https://github.com/pnngocdoan/agent-history-effects {REPO}
%cd {REPO}
!git pull
sys.path.insert(0, REPO)

!pip -q install vllm

In [ ]:
# 2. access (once per runtime; tokens are never stored or printed)
from getpass import getpass
import subprocess

name = input('Your name (for commit messages): ')
email = input('Your GitHub email (or <username>@users.noreply.github.com): ')
token = getpass('GitHub token (repo scope): ')
subprocess.run(['git', 'config', 'user.name', name], cwd=REPO)
subprocess.run(['git', 'config', 'user.email', email], cwd=REPO)
subprocess.run(['git', 'remote', 'set-url', 'origin',
                f'https://{token}@github.com/pnngocdoan/agent-history-effects.git'], cwd=REPO)
del token

hf = getpass('Hugging Face token (leave empty if no model is gated): ')
if hf:
    os.environ['HF_TOKEN'] = hf
del hf

def push(msg):
    r = subprocess.run(f'git add results reports/setup.json && git commit -m "{msg}" && git pull --rebase && git push',
                       shell=True, cwd=REPO, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip().splitlines()
    print(out[-1] if out else 'nothing to push')

In [ ]:
# 3. background saver: pushes results every 10 minutes while the run goes, so a dead
#    runtime loses at most 10 minutes of episodes
import threading, time

def _saver():
    while not _stop.is_set():
        _stop.wait(600)
        push('autosave: main study in progress')

_stop = threading.Event()
threading.Thread(target=_saver, daemon=True).start()
print('autosave armed (every 10 min)')

In [ ]:
# 4. THE RUN — keep only your model(s) in MODELS; rerunning only fills in what's missing
import json, time
from pathlib import Path
import serve

#          family     Hugging Face id      short name (results folder and viewer tab)
MODELS = [('Qwen3',   'Qwen/Qwen3-0.6B',   'qwen3-0.6b'),
          ('Qwen3',   'Qwen/Qwen3-1.7B',   'qwen3-1.7b'),
          ('Qwen3',   'Qwen/Qwen3-4B',     'qwen3-4b'),
          ('Qwen3',   'Qwen/Qwen3-8B-AWQ', 'qwen3-8b')]
ROUNDS, EPISODES = '0 10', 10      # k = 0 (baseline, no junk rounds) and k = 10; conversations per k
TARGET = 2 * EPISODES

def n_done(out):
    f = Path(out) / 'episodes.jsonl'
    return sum(1 for l in f.read_text().splitlines() if l.strip()) if f.exists() else 0

def pilot_problems(out):
    """3 conversations at k = 10: does the model call tools, finish its turns, and act without asking?"""
    rows = [json.loads(l) for l in Path(out, 'episodes.jsonl').read_text().splitlines() if l.strip()]
    calls = sum(1 for r in rows for m in r['messages'] if m.get('tool_calls'))
    problems = []
    if len(rows) < 3: problems.append(f'only {len(rows)}/3 conversations finished')
    if calls == 0: problems.append('no tool calls at all: check the tool-call parser for this family in serve.py')
    if sum(r['truncated'] for r in rows): problems.append('turns were cut off at the token limit')
    if rows and sum(r['asked'] for r in rows) == len(rows): problems.append('every conversation ended by asking the user')
    return problems

for family, hf, name in MODELS:
    out = f'results/multiturn/{name}'
    if n_done(out) >= TARGET:
        print(f'{name}: complete, skipping'); continue
    print(f'=== {name} ({family}) ===', flush=True)
    temp = serve.FAMILIES[family]['temperature']
    !pkill -f api_server
    time.sleep(8)
    proc, log = serve.start_server(hf, name, log_dir=f'results/logs/{name}', family=family)
    serve.wait_for_server(proc, log=log)
    pilot = f'results/pilot/{name}'
    !python multiturn/run_pilot.py --model {name} --family "{family}" --rounds 10 --episodes 3 \
        --temperature {temp} --workers 3 --out {pilot}
    problems = pilot_problems(pilot)
    if problems:
        print(f'{name}: pilot failed, skipping the full run:\n- ' + '\n- '.join(problems))
        proc.terminate(); push(f'pilot failed: {name}'); continue
    !python multiturn/run_pilot.py --model {name} --family "{family}" --rounds {ROUNDS} \
        --episodes {EPISODES} --temperature {temp} --workers 8 --out {out}
    proc.terminate()
    push(f'results: {name}')      # saved to GitHub before the next model starts
_stop.set()
push('main study run complete')
print('done — results on GitHub; open reports/viewer.html to inspect')

In [ ]:
# 5. tables — per model and across models
from pathlib import Path
from multiturn.review import review_summary, summarize_results

for _, _, name in MODELS:
    out = f'results/multiturn/{name}'
    if Path(out, 'episodes.jsonl').exists():
        print(f'\n===== {name} =====')
        review_summary(out, model=name)
summarize_results('results/multiturn')

## Inspecting episodes

`reports/viewer.html` fetches the pushed `results/.../episodes.jsonl` in the browser: nothing
to regenerate, ever. Serve it over http:

- **GitHub Pages**: repo Settings → Pages → deploy from `main`, then open
  `https://<user>.github.io/agent-history-effects/reports/viewer.html`
- **locally**: `python -m http.server` in the repo root → `http://localhost:8000/reports/viewer.html`

Edit the HTML (colors, layout, filters), push, refresh — the experiments never rerun.

In [ ]:
# 6. (optional) hand-run single episodes outside the sweep
# from multiturn.review import run_and_review
# run_and_review(k=3, model='qwen3-1.7b', out='results/multiturn/qwen3-1.7b')